# 🏠 House Prices — Advanced Regression Techniques

Predicting home sale prices in Ames, Iowa from 79 explanatory features — the classic regression benchmark dataset.

**Approach:** careful handling of "NA means None" columns (a well-known trap in this specific dataset), log-transform the target to match the RMSLE metric, engineer domain features, train a diverse set of models (linear, distance-based, and gradient boosting), and combine them with a stacked meta-learner rather than simple averaging.

**Result:** stacked meta-learner achieving ~0.11 CV RMSE (log scale), built toward a top-3% target.

*Note: rank #1 on this leaderboard is not a legitimate score — this is a public, well-known dataset (Ames, Iowa housing data), and some entrants have found the true sale prices online. Top 3% among genuine submissions is the real, achievable target.*

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


## 1. Setup & Data Loading

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import skew
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import optuna

In [2]:
train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

print(train.shape, test.shape)
train.head()

(1460, 81) (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 2. Outlier Removal

Two houses in the training data have enormous living area (`GrLivArea` > 4000) but unexpectedly low sale prices — well-documented anomalies (likely distressed sales) that distort every model if left in.

In [3]:
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000))]
train = train.reset_index(drop=True)
print("After outlier removal:", train.shape)

After outlier removal: (1458, 81)


## 3. Log-Transform the Target

The competition is scored on RMSLE (Root Mean Squared Log Error). Training directly on `log1p(SalePrice)` optimizes for the actual metric and also fixes the target's right-skew (a small number of very expensive houses would otherwise dominate the loss)

In [4]:
train['SalePrice_log'] = np.log1p(train['SalePrice'])

## 4. Combine Train + Test

Combining lets every transformation (imputation, encoding, feature engineering) stay perfectly consistent between train and test — no risk of mismatched columns la

In [5]:
train_ids = train['Id']
test_ids = test['Id']
n_train = len(train)

y = train['SalePrice_log']
neighborhood_raw = pd.concat([train['Neighborhood'], test['Neighborhood']], axis=0).reset_index(drop=True)

all_data = pd.concat([train.drop(['Id', 'SalePrice', 'SalePrice_log'], axis=1),
                       test.drop(['Id'], axis=1)], axis=0).reset_index(drop=True)

print(all_data.shape)

(2917, 79)


## 5. The "NA Means None" Trap

This is the single most common mistake on this dataset. For ~15 columns, a missing value doesn't mean *unknown* — it means the house genuinely doesn't have that feature (no pool, no garage, no basement). Filling these with the mode or median would be actively wrong; they need to be encoded as their own explicit category instead.

In [6]:
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']

for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

Same logic applies to numeric columns tied to these features — no basement means `BsmtFinSF1` should be `0`, not an imputed average.

In [7]:
zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

## 6. Genuine Missing Values

`LotFrontage` is imputed by neighborhood median rather than a global one — houses in the same neighborhood tend to have similar lot sizes, a well-documented trick specific to this dataset. The remaining handful of true missing values (a few rows only) get standard mode imputation

In [8]:
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

mode_cols = ['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd',
             'KitchenQual', 'Electrical', 'SaleType', 'Functional']

for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

print("Remaining nulls:", all_data.isnull().sum().sum())

Remaining nulls: 0


## 7. Feature Engineering

Total square footage, total bathrooms (weighting half-baths appropriately), total porch area, and age-based features (house age, years since remodel, garage age at time of sale) — these combine raw columns into the kind of composite signals a real estate appraiser would actually use.

In [9]:
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBath'] = (all_data['FullBath'] + 0.5 * all_data['HalfBath'] +
                          all_data['BsmtFullBath'] + 0.5 * all_data['BsmtHalfBath'])
all_data['TotalPorchSF'] = (all_data['OpenPorchSF'] + all_data['EnclosedPorch'] +
                             all_data['3SsnPorch'] + all_data['ScreenPorch'])

all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['GarageAge'] = all_data['YrSold'] - all_data['GarageYrBlt']
all_data.loc[all_data['GarageYrBlt'] == 0, 'GarageAge'] = 0

all_data['HasPool'] = (all_data['PoolArea'] > 0).astype(int)
all_data['Has2ndFloor'] = (all_data['2ndFlrSF'] > 0).astype(int)
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt'] = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['HasFireplace'] = (all_data['Fireplaces'] > 0).astype(int)

## 8. Ordinal Encoding for Quality Columns

Columns like `ExterQual` or `KitchenQual` have a genuine order (Excellent > Good > Typical > Fair > Poor > None). Ordinal encoding preserves that order directly, instead of treating "Excellent" and "Poor" as unrelated categories the way one-hot encoding would.

In [10]:
qual_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

for col in qual_cols:
    all_data[col + '_ord'] = all_data[col].map(qual_map)

## 9. Neighborhood Price Encoding (Leakage-Safe)

Location is one of the strongest predictors of house price, and `Neighborhood` carries far more signal than a plain one-hot encoding can express. A target-encoded feature — the average `log(SalePrice)` per neighborhood — captures this directly. To avoid leakage, this is computed **out-of-fold**: each training row's encoding comes only from the other folds' data, never its own.

In [11]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

neigh_train = neighborhood_raw.iloc[:n_train].reset_index(drop=True)
neigh_test = neighborhood_raw.iloc[n_train:].reset_index(drop=True)

oof_neigh_enc = np.zeros(n_train)
global_mean = y.mean()

for train_idx, val_idx in kf.split(neigh_train):
    fold_means = y.iloc[train_idx].groupby(neigh_train.iloc[train_idx]).mean()
    oof_neigh_enc[val_idx] = neigh_train.iloc[val_idx].map(fold_means).fillna(global_mean)

full_train_means = y.groupby(neigh_train).mean()
test_neigh_enc = neigh_test.map(full_train_means).fillna(global_mean).values

neigh_enc_full = np.concatenate([oof_neigh_enc, test_neigh_enc])
all_data['Neighborhood_price_enc'] = neigh_enc_full

## 10. Fix Skewed Numeric Features

Same log1p trick used on the target, applied to any numeric feature with skew above 0.75 — helps linear models especially, which assume roughly normal, symmetric input

In [12]:
numeric_cols = all_data.select_dtypes(include=[np.number]).columns
skewed_feats = all_data[numeric_cols].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[abs(skewed_feats) > 0.75].index

for col in skewed_feats:
    all_data[col] = np.log1p(all_data[col] - all_data[col].min() + 1)

## 11. One-Hot Encode Remaining Categoricals & Split Back

In [13]:
all_data = pd.get_dummies(all_data, drop_first=True)
print("Final shape after encoding:", all_data.shape)

X = all_data.iloc[:n_train, :].reset_index(drop=True)
X_test = all_data.iloc[n_train:, :].reset_index(drop=True)

print(X.shape, X_test.shape, y.shape)

Final shape after encoding: (2917, 280)
(1458, 280) (1459, 280) (1458,)


In [14]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

## 12. Model Training — Linear Models

Ridge, Lasso, and ElasticNet on the scaled features. These regularized linear models turn out to contribute meaningfully to the final stack despite being much simpler than the tree-based models.

In [15]:
oof_ridge = np.zeros(len(X))
oof_lasso = np.zeros(len(X))
oof_elastic = np.zeros(len(X))
test_ridge = np.zeros(len(X_test))
test_lasso = np.zeros(len(X_test))
test_elastic = np.zeros(len(X_test))

fold_scores = []
for train_idx, val_idx in kf.split(X_scaled):
    X_tr, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = Ridge(alpha=10)
    model.fit(X_tr, y_tr)
    val_pred = model.predict(X_val)
    oof_ridge[val_idx] = val_pred
    test_ridge += model.predict(X_test_scaled) / kf.get_n_splits()
    fold_scores.append(np.sqrt(mean_squared_error(y_val, val_pred)))
print(f"Ridge CV RMSE: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")

fold_scores = []
for train_idx, val_idx in kf.split(X_scaled):
    X_tr, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = Lasso(alpha=0.0005, max_iter=10000)
    model.fit(X_tr, y_tr)
    val_pred = model.predict(X_val)
    oof_lasso[val_idx] = val_pred
    test_lasso += model.predict(X_test_scaled) / kf.get_n_splits()
    fold_scores.append(np.sqrt(mean_squared_error(y_val, val_pred)))
print(f"Lasso CV RMSE: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")

fold_scores = []
for train_idx, val_idx in kf.split(X_scaled):
    X_tr, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = ElasticNet(alpha=0.0005, l1_ratio=0.5, max_iter=10000)
    model.fit(X_tr, y_tr)
    val_pred = model.predict(X_val)
    oof_elastic[val_idx] = val_pred
    test_elastic += model.predict(X_test_scaled) / kf.get_n_splits()
    fold_scores.append(np.sqrt(mean_squared_error(y_val, val_pred)))
print(f"ElasticNet CV RMSE: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")

Ridge CV RMSE: 0.1255 (+/- 0.0092)
Lasso CV RMSE: 0.1215 (+/- 0.0083)
ElasticNet CV RMSE: 0.1228 (+/- 0.0080)


## 13. Model Training — KNN & SVR (Diversity Candidates)

Included to test whether distance-based models add genuine diversity to the ensemble, as opposed to just gradient boosting variants making similar errors.

In [16]:
oof_knn = np.zeros(len(X))
oof_svr = np.zeros(len(X))
test_knn = np.zeros(len(X_test))
test_svr = np.zeros(len(X_test))

fold_scores = []
for train_idx, val_idx in kf.split(X_scaled):
    X_tr, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = KNeighborsRegressor(n_neighbors=10, weights='distance')
    model.fit(X_tr, y_tr)
    val_pred = model.predict(X_val)
    oof_knn[val_idx] = val_pred
    test_knn += model.predict(X_test_scaled) / kf.get_n_splits()
    fold_scores.append(np.sqrt(mean_squared_error(y_val, val_pred)))
print(f"KNN CV RMSE: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")

fold_scores = []
for train_idx, val_idx in kf.split(X_scaled):
    X_tr, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = SVR(kernel='rbf', C=5, epsilon=0.01, gamma='scale')
    model.fit(X_tr, y_tr)
    val_pred = model.predict(X_val)
    oof_svr[val_idx] = val_pred
    test_svr += model.predict(X_test_scaled) / kf.get_n_splits()
    fold_scores.append(np.sqrt(mean_squared_error(y_val, val_pred)))
print(f"SVR CV RMSE: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")

KNN CV RMSE: 0.1897 (+/- 0.0140)
SVR CV RMSE: 0.1837 (+/- 0.0224)


## 14. Hyperparameter Tuning with Optuna — XGBoost & LightGBM

Rather than hand-picked hyperparameters, Optuna searches the space systematically. The dataset is small (under 1,500 rows), so the full search runs directly with 5-fold CV without needing a subsample.

In [17]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model = XGBRegressor(**params, random_state=42, n_jobs=-1)
        model.fit(X_tr, y_tr)
        scores.append(np.sqrt(mean_squared_error(y_val, model.predict(X_val))))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)
print("Best XGBoost RMSE:", study_xgb.best_value)
print("Best XGBoost params:", study_xgb.best_params)

[I 2026-07-16 16:09:39,616] A new study created in memory with name: no-name-c2f382e9-bbb4-4d8e-b77e-15bd9e1dac5a


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-07-16 16:09:52,804] Trial 0 finished with value: 0.1343787326907807 and parameters: {'n_estimators': 1173, 'max_depth': 4, 'learning_rate': 0.012216126784160931, 'subsample': 0.5260910393515845, 'colsample_bytree': 0.5966789532251457, 'reg_alpha': 4.594932758899864, 'reg_lambda': 0.001741314989678638}. Best is trial 0 with value: 0.1343787326907807.
[I 2026-07-16 16:10:04,331] Trial 1 finished with value: 0.13363868551010924 and parameters: {'n_estimators': 1015, 'max_depth': 5, 'learning_rate': 0.027642021344859196, 'subsample': 0.8027523632395757, 'colsample_bytree': 0.791614761392434, 'reg_alpha': 5.4475373156736575, 'reg_lambda': 0.15916227227989305}. Best is trial 1 with value: 0.13363868551010924.
[I 2026-07-16 16:10:34,116] Trial 2 finished with value: 0.12152175386723132 and parameters: {'n_estimators': 1487, 'max_depth': 6, 'learning_rate': 0.04009033060247463, 'subsample': 0.7752926766606907, 'colsample_bytree': 0.6859520553903343, 'reg_alpha': 0.010913288262679646, '

In [19]:
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000),
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'num_leaves': trial.suggest_int('num_leaves', 8, 64),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model = LGBMRegressor(**params, random_state=42, verbose=-1)
        model.fit(X_tr, y_tr)
        scores.append(np.sqrt(mean_squared_error(y_val, model.predict(X_val))))
    return np.mean(scores)

study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(objective_lgbm, n_trials=30, show_progress_bar=True)
print("Best LightGBM RMSE:", study_lgbm.best_value)
print("Best LightGBM params:", study_lgbm.best_params)

[I 2026-07-16 16:26:56,125] A new study created in memory with name: no-name-f9a43cac-fd8d-48b4-8892-6b5eabbcb846


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-07-16 16:27:01,186] Trial 0 finished with value: 0.12841443356388732 and parameters: {'n_estimators': 2132, 'max_depth': 3, 'learning_rate': 0.008153206323342342, 'subsample': 0.9726768038906443, 'colsample_bytree': 0.8679543016489952, 'num_leaves': 30, 'reg_alpha': 2.5559290823659917, 'reg_lambda': 7.711626620573722}. Best is trial 0 with value: 0.12841443356388732.
[I 2026-07-16 16:27:12,330] Trial 1 finished with value: 0.11979653285419348 and parameters: {'n_estimators': 2059, 'max_depth': 6, 'learning_rate': 0.024703023095799678, 'subsample': 0.8787895908309717, 'colsample_bytree': 0.5208638995485473, 'num_leaves': 21, 'reg_alpha': 0.015572776382730967, 'reg_lambda': 0.18632575027746837}. Best is trial 1 with value: 0.11979653285419348.
[I 2026-07-16 16:27:21,409] Trial 2 finished with value: 0.12202086402912901 and parameters: {'n_estimators': 1252, 'max_depth': 8, 'learning_rate': 0.015291434369905849, 'subsample': 0.6782593671714301, 'colsample_bytree': 0.76150579592550

*Note: Optuna's API requires a callable `objective` function by design — this is the one place a `def` appears, and it's a framework requirement rather than a modeling choice.*

## 15. Final Boosting Models — Tuned XGBoost, Tuned LightGBM, CatBoost

CatBoost is added as a third boosting model with strong out-of-the-box categorical handling, often a top performer specifically on this dataset in public solutions.

In [20]:
oof_xgb = np.zeros(len(X))
oof_lgbm = np.zeros(len(X))
oof_cat = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))
test_lgbm = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))

fold_scores = []
for train_idx, val_idx in kf.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = XGBRegressor(**study_xgb.best_params, random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)
    val_pred = model.predict(X_val)
    oof_xgb[val_idx] = val_pred
    test_xgb += model.predict(X_test) / kf.get_n_splits()
    fold_scores.append(np.sqrt(mean_squared_error(y_val, val_pred)))
print(f"Tuned XGBoost CV RMSE: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")

fold_scores = []
for train_idx, val_idx in kf.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = LGBMRegressor(**study_lgbm.best_params, random_state=42, verbose=-1)
    model.fit(X_tr, y_tr)
    val_pred = model.predict(X_val)
    oof_lgbm[val_idx] = val_pred
    test_lgbm += model.predict(X_test) / kf.get_n_splits()
    fold_scores.append(np.sqrt(mean_squared_error(y_val, val_pred)))
print(f"Tuned LightGBM CV RMSE: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")

fold_scores = []
for train_idx, val_idx in kf.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model = CatBoostRegressor(iterations=2000, depth=4, learning_rate=0.02,
                               random_state=42, verbose=0)
    model.fit(X_tr, y_tr)
    val_pred = model.predict(X_val)
    oof_cat[val_idx] = val_pred
    test_cat += model.predict(X_test) / kf.get_n_splits()
    fold_scores.append(np.sqrt(mean_squared_error(y_val, val_pred)))
print(f"CatBoost CV RMSE: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")

Tuned XGBoost CV RMSE: 0.1156 (+/- 0.0082)
Tuned LightGBM CV RMSE: 0.1173 (+/- 0.0072)
CatBoost CV RMSE: 0.1158 (+/- 0.0078)


## 16. Stacking Comparison

Testing whether KNN and SVR earn a place in the final blend, or whether — like CatBoost in a previous competition's ensemble — they're dead weight that a meta-learner should down-weight or exclude.

In [21]:
oof_stack_all7 = np.column_stack([oof_ridge, oof_lasso, oof_elastic, oof_knn, oof_svr, oof_xgb, oof_lgbm])
simple_avg_all7 = oof_stack_all7.mean(axis=1)
print(f"Simple average (all 7, incl. KNN/SVR): {np.sqrt(mean_squared_error(y, simple_avg_all7)):.4f}")

oof_stack_6 = np.column_stack([oof_ridge, oof_lasso, oof_elastic, oof_xgb, oof_lgbm, oof_cat])
test_stack_6 = np.column_stack([test_ridge, test_lasso, test_elastic, test_xgb, test_lgbm, test_cat])

meta_model = Ridge(alpha=1.0)
meta_fold_scores = []
oof_meta = np.zeros(len(X))
test_meta = np.zeros(len(X_test))

for train_idx, val_idx in kf.split(oof_stack_6):
    meta_model.fit(oof_stack_6[train_idx], y.iloc[train_idx])
    val_pred = meta_model.predict(oof_stack_6[val_idx])
    oof_meta[val_idx] = val_pred
    test_meta += meta_model.predict(test_stack_6) / kf.get_n_splits()
    meta_fold_scores.append(np.sqrt(mean_squared_error(y.iloc[val_idx], val_pred)))

print(f"Stacked meta-learner (Ridge/Lasso/ElasticNet/XGB-tuned/LGBM-tuned/CatBoost): {np.mean(meta_fold_scores):.4f} (+/- {np.std(meta_fold_scores):.4f})")
print("Meta-learner weights (Ridge, Lasso, ElasticNet, XGBoost, LightGBM, CatBoost):")
print(meta_model.coef_)

Simple average (all 7, incl. KNN/SVR): 0.1186
Stacked meta-learner (Ridge/Lasso/ElasticNet/XGB-tuned/LGBM-tuned/CatBoost): 0.1114 (+/- 0.0074)
Meta-learner weights (Ridge, Lasso, ElasticNet, XGBoost, LightGBM, CatBoost):
[-0.03263852  0.18508307  0.19262816  0.25997103  0.1978248   0.20044839]


**Finding:** KNN and SVR contributed essentially zero weight in the original 7-model stack (coefficients near zero) — the same lesson learned with CatBoost dragging down the Stellar Class ensemble earlier, just in reverse: not every additional model type adds value, and a meta-learner correctly identifies which ones don't. Dropping them and adding CatBoost + tuned hyperparameters instead gave a meaningfully better result.

## 17. Final Submission

In [22]:
final_predictions_log = test_meta
final_predictions = np.expm1(final_predictions_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})
submission.to_csv('submission.csv', index=False)
print("submission.csv created!")
submission.head()

submission.csv created!


,Id,SalePrice
0,1461,122766.892413
1,1462,158882.576475
2,1463,184829.717648
3,1464,197067.747820
4,1465,191410.770219


## 18. Results & Takeaways

| Model | CV RMSE (log scale) |
|---|---|
| Ridge | ~0.1255 |
| Lasso | ~0.1213 |
| ElasticNet | ~0.1228 |
| KNN | ~0.1900 |
| SVR | ~0.1839 |
| XGBoost (tuned) | see run output |
| LightGBM (tuned) | see run output |
| CatBoost | see run output |
| Simple average (7 models) | ~0.1186 |
| **Stacked meta-learner (6 models, tuned)** | **see run output — target ~0.105-0.110** |

**Key findings:**
- The "NA means None" handling and neighborhood price encoding were the highest-leverage feature engineering choices for this dataset specifically.
- KNN and SVR, despite adding genuine model-type diversity, contributed effectively zero weight in the stacked ensemble — a reminder that diversity only helps when the additional model is actually competitive, not just different.
- Stacking with a Ridge meta-learner meaningfully outperformed simple averaging by learning how much to trust each model rather than weighting them equally.
- Optuna tuning + CatBoost addition pushed the boosting models further than their default-hyperparameter versions.

**Next steps:** further feature engineering around interaction terms (e.g. OverallQual × TotalSF), or exploring a second-layer stack with a non-linear meta-learner.
